In [2]:
import pandas as pd

movies = pd.read_csv(
    "data/movies.csv",
    # sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
    skiprows=1,
    header=None
)

print(movies.head())

   movie_id                               title  \
0         1                    Toy Story (1995)   
1         2                      Jumanji (1995)   
2         3             Grumpier Old Men (1995)   
3         4            Waiting to Exhale (1995)   
4         5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [3]:
movies["genres"] = movies["genres"].apply(lambda x: x.split("|"))

In [4]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["title"] = movies["title"].str.replace(r"\(\d{4}\)", "", regex=True).str.strip()

In [5]:
movies

,movie_id,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017
9739,193585,Flint,[Drama],2017
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018


Ratings.dat 읽기

In [6]:
ratings = pd.read_csv(
    "data/ratings.csv",
    # sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    skiprows=1,
    header=None
)

ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

In [7]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [8]:
# avg_ratings = ratings.groupby("movie_id")["rating"]..mean().reset_index()
avg_ratings = (
    ratings.groupby("movie_id")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)
avg_ratings.rename(columns={"rating": "avg_rating"}, inplace=True)

In [9]:
avg_ratings

,movie_id,avg_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49
...,...,...,...
9719,193581,4.000000,1
9720,193583,3.500000,1
9721,193585,3.500000,1
9722,193587,3.500000,1


In [10]:
movies = movies.merge(avg_ratings, on="movie_id", how="left")

In [11]:
movies

,movie_id,title,genres,year,avg_rating,rating_count
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,3.920930,215.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,3.431818,110.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,3.259615,52.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,2.357143,7.0
4,5,Father of the Bride Part II,[Comedy],1995,3.071429,49.0
...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017,4.000000,1.0
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017,3.500000,1.0
9739,193585,Flint,[Drama],2017,3.500000,1.0
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018,3.500000,1.0


In [12]:
tags = pd.read_csv(
    "data/tags.csv",
    # sep="::",
    engine="python",
    names=["user_id", "movie_id", "tag", "timestamp"],
    skiprows=1,
    header=None
)

tags["tag"] =tags["tag"].str.lower().str.strip()
tags = tags[tags["tag"] != ""]
tags = tags.dropna(subset=["movie_id", "tag"])
tags["tag"].apply(type).value_counts()

tag
<class 'str'>    3683
Name: count, dtype: int64

In [13]:
tags_agg = (tags.groupby("movie_id")["tag"].apply(lambda x: " | ".join(sorted(set(x)))).reset_index().rename(columns={"tag": "tags"}))

tags_agg

,movie_id,tags
0,1,fun | pixar
1,2,fantasy | game | magic board game | robin will...
2,3,moldy | old
3,5,pregnancy | remake
4,7,remake
...,...,...
1567,183611,comedy | funny | rachel mcadams
1568,184471,adventure | alicia vikander | video game adapt...
1569,187593,josh brolin | ryan reynolds | sarcasm
1570,187595,emilia clarke | star wars


In [14]:
tags_agg["tags"] = tags_agg["tags"].apply(lambda x: x.split(" | "))

In [15]:
movies = movies.merge(tags_agg, on="movie_id", how="left")

In [16]:
movies


,movie_id,title,genres,year,avg_rating,rating_count,tags
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,3.920930,215.0,"[fun, pixar]"
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,3.431818,110.0,"[fantasy, game, magic board game, robin williams]"
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,3.259615,52.0,"[moldy, old]"
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,2.357143,7.0,NaN
4,5,Father of the Bride Part II,[Comedy],1995,3.071429,49.0,"[pregnancy, remake]"
...,...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017,4.000000,1.0,NaN
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017,3.500000,1.0,NaN
9739,193585,Flint,[Drama],2017,3.500000,1.0,NaN
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018,3.500000,1.0,NaN


# TMDb 데이터 붙이기

In [17]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TMDB_BEARER_TOKEN = os.getenv("TMDB_BEARER_TOKEN")
BASE_URL = "https://api.themoviedb.org/3"

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_BEARER_TOKEN}",
}

c:\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [18]:
def tmdb_search_movie(query: str, year=None, language="en-US"):
    url = f"{BASE_URL}/search/movie"
    params = {
        "query": query,
        "language": language,
        "include_adult": "false",
    }
    if year:
        params["year"] = year

    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])
        if not results:
            return None

        # 간단한 우선순위: 정확한 제목 + 연도 근접 + popularity 높은 순
        q = query.lower().strip()

        def score(item):
            title = str(item.get("title", "")).lower().strip()
            original_title = str(item.get("original_title", "")).lower().strip()
            exact = 1 if (title == q or original_title == q) else 0

            release_date = item.get("release_date", "")
            item_year = None
            if isinstance(release_date, str) and len(release_date) >= 4:
                try:
                    item_year = int(release_date[:4])
                except:
                    item_year = None

            year_score = 0
            if year and item_year:
                year_score = -abs(year - item_year)

            popularity = float(item.get("popularity", 0))
            return (exact, year_score, popularity)

        best = sorted(results, key=score, reverse=True)[0]
        return best

    except requests.RequestException as e:
        print(f"[SEARCH ERROR] {query} ({year}): {e}")
        return None

In [19]:
def tmdb_get_movie_details(tmdb_id: int, language="en-US"):
    url = f"{BASE_URL}/movie/{tmdb_id}"
    params = {
        "language": language,
        "append_to_response": "credits,keywords"
    }

    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
        resp.raise_for_status()
        return resp.json()
    except requests.RequestException as e:
        print(f"[DETAIL ERROR] {tmdb_id}: {e}")
        return None

In [20]:
import time

def enrich_movies_with_tmdb(df, max_rows=10, sleep_sec=0.25):
    df = df.copy()

    # 결과 컬럼 미리 생성
    add_cols = [
        "tmdb_id", "tmdb_title", "overview", "runtime", "release_date",
        "tmdb_vote_average", "tmdb_vote_count", "director", "cast_top3",
        "tmdb_genres", "poster_path", "popularity"
    ]
    for c in add_cols:
        if c not in df.columns:
            df[c] = None

    target_idx = df.index[:max_rows] if max_rows else df.index

    for idx in target_idx:
        row = df.loc[idx]
        title = row["title"]
        year = int(row["year"])

        if not title:
            continue
        print ("title", title)
        search_result = tmdb_search_movie(title, year=year)
        time.sleep(sleep_sec)

        if not search_result:
            print(f"[NO MATCH] {title} ({year})")
            continue

        tmdb_id = search_result.get("id")
        if not tmdb_id:
            print(f"[NO TMDB ID] {title} ({year})")
            continue

        details = tmdb_get_movie_details(tmdb_id)
        time.sleep(sleep_sec)

        if not details:
            continue

        credits = details.get("credits", {})
        cast = credits.get("cast", []) if isinstance(credits, dict) else []
        crew = credits.get("crew", []) if isinstance(credits, dict) else []

        director = None
        for member in crew:
            if member.get("job") == "Director":
                director = member.get("name")
                break

        cast_top3 = [c.get("name") for c in cast[:3] if c.get("name")]
        genre_names = [g.get("name") for g in details.get("genres", []) if g.get("name")]

        df.at[idx, "tmdb_id"] = details.get("id")
        df.at[idx, "tmdb_title"] = details.get("title")
        df.at[idx, "overview"] = details.get("overview")
        df.at[idx, "runtime"] = details.get("runtime")
        df.at[idx, "release_date"] = details.get("release_date")
        df.at[idx, "tmdb_vote_average"] = details.get("vote_average")
        df.at[idx, "tmdb_vote_count"] = details.get("vote_count")
        df.at[idx, "director"] = director
        df.at[idx, "cast_top3"] = "|".join(cast_top3) if cast_top3 else None
        df.at[idx, "tmdb_genres"] = "|".join(genre_names) if genre_names else None
        df.at[idx, "poster_path"] = details.get("poster_path")
        df.at[idx, "popularity"] = details.get("popularity")

        print(f"[OK] {row['movie_id']} | {title} ({year}) -> TMDb {tmdb_id}")

    return df

In [ ]:
movies_enriched = enrich_movies_with_tmdb(movies, max_rows=100, sleep_sec=0.3)
movies_enriched[
    ["movie_id", "title", "year", "tmdb_title", "overview", "runtime", "director"]
].head(10)

title Toy Story
[OK] 1 | Toy Story (1995) -> TMDb 862
title Jumanji
[OK] 2 | Jumanji (1995) -> TMDb 8844
title Grumpier Old Men
[OK] 3 | Grumpier Old Men (1995) -> TMDb 15602
title Waiting to Exhale
[OK] 4 | Waiting to Exhale (1995) -> TMDb 31357
title Father of the Bride Part II
[OK] 5 | Father of the Bride Part II (1995) -> TMDb 11862
title Heat
[OK] 6 | Heat (1995) -> TMDb 949
title Sabrina
[OK] 7 | Sabrina (1995) -> TMDb 11860
title Tom and Huck


KeyboardInterrupt: 

LangChain Document 생성

In [ ]:
from langchain_core.documents import Document

docs = []

for _, row in movies_enriched.iterrows():
    page_content = f"""
Movie title: {row.get('tmdb_title') or row.get('title_clean')}
Release year: {row.get('year')}
Genres: {row.get('tmdb_genres') or ', '.join(row.get('genres_list', []))}
Overview: {row.get('overview') or ''}
Director: {row.get('director') or ''}
Cast: {row.get('cast_top3') or ''}
Average rating: {row.get('avg_rating', '')}
Rating count: {row.get('rating_count', '')}
Runtime: {row.get('runtime', '')} minutes
"""

    metadata = {
        "movieId": row.get("movieId"),
        "title": row.get("tmdb_title") or row.get("title_clean"),
        "year": row.get("year"),
        "genres": row.get("genres_list"),
        "avg_rating": row.get("avg_rating"),
        "rating_count": row.get("rating_count"),
        "runtime": row.get("runtime"),
        "director": row.get("director"),
    }

    docs.append(Document(page_content=page_content, metadata=metadata))

In [ ]:
docs

[Document(metadata={'movie_id': 1, 'title': 'Toy Story', 'year': '1995', 'genres': ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'], 'rating': 3.9209302325581397}, page_content='\nMovie title: Toy Story\nRelease year: 1995\nGenres: Adventure, Animation, Children, Comedy, Fantasy\nAverage rating: 3.9209302325581397\n'),
 Document(metadata={'movie_id': 2, 'title': 'Jumanji', 'year': '1995', 'genres': ['Adventure', 'Children', 'Fantasy'], 'rating': 3.4318181818181817}, page_content='\nMovie title: Jumanji\nRelease year: 1995\nGenres: Adventure, Children, Fantasy\nAverage rating: 3.4318181818181817\n'),
 Document(metadata={'movie_id': 3, 'title': 'Grumpier Old Men', 'year': '1995', 'genres': ['Comedy', 'Romance'], 'rating': 3.2596153846153846}, page_content='\nMovie title: Grumpier Old Men\nRelease year: 1995\nGenres: Comedy, Romance\nAverage rating: 3.2596153846153846\n'),
 Document(metadata={'movie_id': 4, 'title': 'Waiting to Exhale', 'year': '1995', 'genres': ['Comedy', 'Dra